In [2]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [3]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
# データフレームを表示して確認
df.head()


,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


In [4]:
# 2. データをLLM用にテキスト形式に変換
# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)

売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


In [5]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

売上データを基に、いくつかの重要な傾向を分析できます。

### 1. カテゴリ別売上の分析
- **食品**、**衣服**、**日用品**という異なるカテゴリーに分かれていますが、売上や数量の分布を調べることで、どのカテゴリーが最も注目を集めているかを把握できます。
- 例えば、食品は価格が比較的低いため、ボリュームが多くなる傾向がありますが、衣服カテゴリは単価が高いため、売上金額が大きい場合があります。

### 2. 商品別売上の分析
- 各商品ごとに売上や利益を分析します。単価と原価の差から利益率を計算し、高い利益率を持つ商品を特定することで、マーケティングや販促活動を集中させるポイントが見えてきます。
- 例えば、ジーンズやTシャツは高価格帯ですが、コストも高いため、販売戦略を見直す必要があります。

### 3. 売上の時間的変化
- 売上日ごとのトレンドを分析することで、特定の時期に売上が突然増加したり減少したりする要因を特定できます。季節性やプロモーション期間が影響を与えることが多いです。
- 例：特定の日（例えば、月初めや祝日）に売上が急増する場合、これらのタイミングを活用したキャンペーンを検討することができます。

### 4. 販売数量と価格の関係
- 数量と単価の関係を分析することで、価格戦略を最適化するためのインサイトが得られます。たとえば、低価格商品は多く売れるが高利益の商品が少量しか売れない場合、バランスを考慮する必要があります。

### 5. 利益分析
- 各商品の利益を計算し、利益額や利益率で上位の商品を特定することが可能です。これにより、利益を最大化するための戦略を立てることができます。
- 原価の高い商品の利益を上げるためのコスト削減や効率化を行う手段を検討することができるでしょう。

### 結論
データ分析から得られるインサイトを基に、以下のアクションを取ることが可能です。
- 高利益で売上が伸び悩んでいる商品に対してのマーケティング戦略策定
- 売れ筋商品をさらに拡充したキャンペーンやプロモーションの実施
- 売上の季節変動を考慮した在庫管理やプロモーションスケジュールの最適化

具体的な数値や詳細な分析を行えば、さらに深い洞察が得られ、データに基づくより効果的なビジネス戦略を策定することができます。


In [6]:
# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])
print(df_out)

                                                   結果
0                         売上データを基に、いくつかの重要な傾向を分析できます。
1                                                    
2                                   ### 1. カテゴリ別売上の分析
3   - **食品**、**衣服**、**日用品**という異なるカテゴリーに分かれていますが、売上...
4   - 例えば、食品は価格が比較的低いため、ボリュームが多くなる傾向がありますが、衣服カテゴリは...
5                                                    
6                                     ### 2. 商品別売上の分析
7   - 各商品ごとに売上や利益を分析します。単価と原価の差から利益率を計算し、高い利益率を持つ商...
8   - 例えば、ジーンズやTシャツは高価格帯ですが、コストも高いため、販売戦略を見直す必要があります。
9                                                    
10                                    ### 3. 売上の時間的変化
11  - 売上日ごとのトレンドを分析することで、特定の時期に売上が突然増加したり減少したりする要因...
12  - 例：特定の日（例えば、月初めや祝日）に売上が急増する場合、これらのタイミングを活用したキ...
13                                                   
14                                  ### 4. 販売数量と価格の関係
15  - 数量と単価の関係を分析することで、価格戦略を最適化するためのインサイトが得られます。たと...
16                                                   
17                          

In [7]:
# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

In [8]:
# ワークフロー化
print("処理を開始します。")

# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')

# 2. データをLLM用にテキスト形式に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"

# 3. OpenAI APIの呼び出し
# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"
# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])

# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

print("Excelファイルに分析結果を保存しました。")

処理を開始します。
Excelファイルに分析結果を保存しました。
